# Notebook 2: HP Lattice Model — Protein Folding Simulation
**ProteinIQ Project**

## Background
The HP (Hydrophobic-Polar) model, introduced by Lau & Dill (1989), reduces protein folding to a combinatorial optimisation problem:
- Each residue is classified as **H** (hydrophobic) or **P** (polar)
- The protein chain is placed on a **2D square lattice**
- Energy = **−1** per non-covalent H-H contact
- Goal: find the conformation **minimising total energy** (maximising H-H contacts)

We solve this using **Monte Carlo Simulated Annealing** — directly applicable to the dynamic molecular simulation concept.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import random, math
np.random.seed(42)
random.seed(42)

## 1. HP Classification of Amino Acids

In [ ]:
HYDROPHOBIC = set('ACFILMVWY')
POLAR       = set('DEHKNQRSTGP')

def to_hp(seq):
    return ''.join('H' if aa in HYDROPHOBIC else 'P' for aa in seq.upper())

# Visualise classification
AA_LIST = list('ACDEFGHIKLMNPQRSTVWY')
colors  = ['#D85A30' if aa in HYDROPHOBIC else '#378ADD' for aa in AA_LIST]

fig, ax = plt.subplots(figsize=(12, 1.5))
for i, (aa, c) in enumerate(zip(AA_LIST, colors)):
    ax.add_patch(plt.Rectangle((i, 0), 1, 1, color=c, alpha=0.8))
    ax.text(i+0.5, 0.5, aa, ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')
ax.set_xlim(0, 20)
ax.set_ylim(0, 1)
ax.axis('off')
patch_H = mpatches.Patch(color='#D85A30', label='Hydrophobic (H)')
patch_P = mpatches.Patch(color='#378ADD', label='Polar (P)')
ax.legend(handles=[patch_H, patch_P], loc='upper right', bbox_to_anchor=(1, 1.5))
ax.set_title('HP classification of the 20 amino acids', pad=20)
plt.tight_layout()
plt.show()

test_seq = 'VLSEGEWQLVLHVWAK'
print(f'Sequence: {test_seq}')
print(f'HP string: {to_hp(test_seq)}')

## 2. Energy Function & Move Operators

In [ ]:
DIRS = [(1,0),(-1,0),(0,1),(0,-1)]

def compute_energy(positions, hp_string):
    """E = -1 for each non-covalent H-H topological contact."""
    pos_dict = {pos: i for i, pos in enumerate(positions)}
    energy   = 0.0
    for i, (x, y) in enumerate(positions):
        if hp_string[i] != 'H':
            continue
        for dx, dy in DIRS:
            nb = (x+dx, y+dy)
            if nb in pos_dict:
                j = pos_dict[nb]
                if abs(j - i) > 1:   # non-covalent
                    energy -= 0.5    # each pair counted twice
    return energy

def extended_chain(n):
    return [(i, 0) for i in range(n)]

print('Extended chain energy (expected ~0):', 
      compute_energy(extended_chain(10), 'HPPHPPHPPH'))

## 3. Monte Carlo Simulated Annealing

In [ ]:
def end_move(positions, idx):
    new = list(positions)
    anchor = new[1] if idx == 0 else new[-2]
    candidates = [(anchor[0]+dx, anchor[1]+dy)
                  for dx,dy in DIRS
                  if (anchor[0]+dx, anchor[1]+dy) not in set(new)]
    if not candidates: return positions
    new[idx] = random.choice(candidates)
    return new

def corner_move(positions, idx):
    new = list(positions)
    if idx <= 0 or idx >= len(new)-1: return positions
    prev, curr, nxt = new[idx-1], new[idx], new[idx+1]
    dx, dy = nxt[0]-prev[0], nxt[1]-prev[1]
    if abs(dx)+abs(dy) != 2: return positions
    nc = (prev[0]+(nxt[0]-prev[0])-(curr[0]-prev[0]),
          prev[1]+(nxt[1]-prev[1])-(curr[1]-prev[1]))
    if nc in set(new): return positions
    new[idx] = nc
    return new

def mc_fold(hp_string, T_start=5.0, T_end=0.01, steps=15000, seed=42):
    random.seed(seed)
    n       = len(hp_string)
    pos     = extended_chain(n)
    E       = compute_energy(pos, hp_string)
    best_E, best_pos = E, list(pos)
    energy_trace, temp_trace = [E], [T_start]
    alpha   = (T_end/T_start)**(1/steps)
    T       = T_start
    accepted = 0

    for step in range(1, steps+1):
        idx    = random.randint(0, n-1)
        mover  = random.choice([end_move, corner_move])
        new_pos = mover(pos, idx)
        if len(set(map(tuple, new_pos))) < n: continue   # self-intersection
        new_E   = compute_energy(new_pos, hp_string)
        dE      = new_E - E
        if dE < 0 or random.random() < math.exp(-dE/max(T, 1e-9)):
            pos, E = new_pos, new_E
            accepted += 1
            if E < best_E:
                best_E, best_pos = E, list(pos)
        T *= alpha
        if step % (steps//100) == 0:
            energy_trace.append(E)
            temp_trace.append(T)

    return {'best_pos': best_pos, 'best_energy': best_E,
            'energy_trace': energy_trace, 'temp_trace': temp_trace,
            'acceptance_rate': accepted/steps}

seq    = 'HPPHPPHPPHHPPHHPPHPP'
result = mc_fold(seq, steps=20000)
print(f'HP string      : {seq}')
print(f'Best energy    : {result["best_energy"]}')
print(f'H-H contacts   : {int(-result["best_energy"]*2)}')
print(f'Acceptance rate: {result["acceptance_rate"]:.3f}')

## 4. Visualise Folded Conformation + Energy Trace

In [ ]:
def plot_conformation(positions, hp_string, title='HP Lattice Conformation'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: lattice plot
    ax = axes[0]
    xs = [p[0] for p in positions]
    ys = [p[1] for p in positions]
    ax.plot(xs, ys, 'k-', linewidth=1.5, zorder=1, alpha=0.4)
    for i, ((x,y), hp) in enumerate(zip(positions, hp_string)):
        color = '#D85A30' if hp == 'H' else '#378ADD'
        circle = plt.Circle((x, y), 0.35, color=color, zorder=2)
        ax.add_patch(circle)
        ax.text(x, y, hp, ha='center', va='center',
                fontsize=9, color='white', fontweight='bold', zorder=3)
        ax.text(x, y-0.55, str(i+1), ha='center', va='center',
                fontsize=7, color='gray')
    # Draw H-H contacts
    pos_dict = {tuple(p): i for i, p in enumerate(positions)}
    for i, (x,y) in enumerate(positions):
        if hp_string[i] != 'H': continue
        for dx,dy in [(1,0),(0,1)]:
            nb = (x+dx, y+dy)
            if nb in pos_dict:
                j = pos_dict[nb]
                if abs(j-i) > 1:
                    ax.plot([x, nb[0]], [y, nb[1]], 'g--', lw=1.5,
                            alpha=0.7, zorder=0)
    ax.set_xlim(min(xs)-1, max(xs)+1)
    ax.set_ylim(min(ys)-1, max(ys)+1)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.set_title(f'{title}\nEnergy = {result["best_energy"]:.1f}  ({int(-result["best_energy"]*2)} H-H contacts)')
    patch_H = mpatches.Patch(color='#D85A30', label='Hydrophobic (H)')
    patch_P = mpatches.Patch(color='#378ADD', label='Polar (P)')
    contact_line = plt.Line2D([0],[0], color='g', ls='--', lw=1.5, label='H-H contact')
    ax.legend(handles=[patch_H, patch_P, contact_line], loc='lower right', fontsize=8)

    # Right: energy + temperature trace
    ax2 = axes[1]
    trace = result['energy_trace']
    x_steps = np.linspace(0, 20000, len(trace))
    ax2.plot(x_steps, trace, color='#1D9E75', linewidth=1.5)
    ax2.fill_between(x_steps, trace, alpha=0.1, color='#1D9E75')
    ax2.axhline(result['best_energy'], color='#D85A30', ls='--', lw=1.2,
                label=f'Best: {result["best_energy"]:.1f}')
    ax2.set_xlabel('MC Step', fontsize=11)
    ax2.set_ylabel('Energy', fontsize=11)
    ax2.set_title('Simulated Annealing Energy Trace')
    ax2.legend()
    ax2.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig('../data/figures/hp_folding.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_conformation(result['best_pos'], seq)

## 5. Effect of Sequence Composition on Foldability

In [ ]:
test_cases = [
    ('HPPHPPHPPHHPPHHPPHPP', 'Alternating HP'),
    ('HHHHHHHHHHHPPPPPPPPP', 'Block H then P'),
    ('HHHPHHPHHPHHPHHPHHPH', 'H-rich'),
    ('PPPPPPPPPPPPPPPPPPPP', 'All polar'),
]
results = []
for hp, label in test_cases:
    r = mc_fold(hp, steps=10000)
    results.append({'label': label, 'hp': hp, 'energy': r['best_energy'],
                    'contacts': int(-r['best_energy']*2)})
    print(f'{label:25s}: E={r["best_energy"]:5.1f}  contacts={int(-r["best_energy"]*2)}')

fig, ax = plt.subplots(figsize=(8, 4))
labels  = [r['label'] for r in results]
energies = [r['energy'] for r in results]
bars = ax.bar(labels, energies, color=['#D85A30','#378ADD','#1D9E75','#888780'], alpha=0.8, edgecolor='none')
ax.set_ylabel('Best energy (lower = more stable)')
ax.set_title('Effect of HP composition on folding energy')
ax.grid(axis='y', alpha=0.3)
for bar, e in zip(bars, energies):
    ax.text(bar.get_x()+bar.get_width()/2, e-0.1, f'{e:.1f}',
            ha='center', va='top', fontsize=10, color='white', fontweight='bold')
plt.tight_layout()
plt.show()